# L3.3 · ACT: el decoder, o sea la policy


### El mapa de toda la serie

ACT es un **VAE condicional**:

- **Encoder del VAE** (sólo al entrenar): lee el estado y el bloque de acciones
  verdadero y lo resume en una variable `z`, el "estilo" de esa demostración.
- **Decoder = la policy** (al entrenar y al usar): un transformer
  encoder-decoder al estilo DETR. Lee la cámara, el estado y la `z`, y escribe
  el bloque entero de acciones de una pasada.
- **Al usarlo:** el encoder del VAE se tira y `z = 0`.

| Notebook | Qué abre |
|---|---|
| L3.1 | Los datos: de la grabación al ejemplo de entrenamiento |
| L3.2 | El encoder del VAE y la `z` |
| L3.3 | El decoder: la policy por dentro |
| L3.4 | El entrenamiento: un paso desarmado, y el modelo aprendiendo |
| L3.5 | El modelo en uso: ejecutar los bloques |

Todo usa el código real de LeRobot 0.4.1 y **nuestro modelo entrenado**
(`act_cheap_v1`), desarmado con `toolkit/policy/act_internals.py`. Sus tests
verifican que las piezas encadenadas dan exactamente lo mismo que el modelo.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy import act_internals as act
from toolkit.policy.load import load_policy

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)
ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
CAMERA = "observation.images.cam_right_high"
loaded = load_policy(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model", device="cpu")
policy, model, config = loaded.policy, loaded.policy.model, loaded.policy.config
_ = policy.eval()

Esta es la parte que se queda: lo que corre en el robot. Es un transformer
**encoder-decoder** al estilo DETR (un detector de objetos de Facebook).

- **Encoder**: mezcla todo lo que se sabe del presente (imagen, estado, `z`)
  en una "memoria".
- **Decoder**: 48 casilleros, uno por paso futuro, que leen esa memoria y se
  convierten en las 48 acciones.

Seguimos una observación real por toda la red.

In [ ]:
plain = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav")
frame = plain[3000]
image = frame[CAMERA]
obs = loaded.preprocessor({k: frame[k] for k in ["observation.state", CAMERA]})
latent = act.zero_latent(model, 1, "cpu")  # al usar el modelo, z = 0
plt.imshow(image.permute(1, 2, 0)); plt.axis("off"); plt.title("la observación que vamos a seguir");

## 1. La imagen → 240 tokens

La ResNet18 (preentrenada en ImageNet) achica la imagen 32 veces en cada lado
y describe cada zona con 512 números: una **grilla de 12×20 zonas**. Con
360×640 da 12×20 = 240; el paper, con 480×640, tenía 15×20 = 300 por cámara.

Después, una convolución 1×1 lleva cada zona de 512 a 192 números (el tamaño
de los tokens).

In [ ]:
with torch.no_grad():
    features = model.backbone(obs[CAMERA])["feature_map"]
    projected = model.encoder_img_feat_input_proj(features)
print("mapa de la ResNet:", tuple(features.shape), "→ proyectado:", tuple(projected.shape))

rows, cols = features.shape[-2:]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].imshow(image.permute(1, 2, 0))
for r in range(rows + 1):
    axes[0].axhline(r * 360 / rows, color="w", lw=0.5)
for c in range(cols + 1):
    axes[0].axvline(c * 640 / cols, color="w", lw=0.5)
axes[0].set_title(f"cada celda es un token ({rows}×{cols} = {rows * cols})"); axes[0].axis("off")
axes[1].imshow(image.permute(1, 2, 0))
axes[1].imshow(features[0].mean(0), cmap="inferno", alpha=0.55, extent=(0, 640, 360, 0), interpolation="bilinear")
axes[1].set_title("activación media de la ResNet por zona"); axes[1].axis("off")
plt.tight_layout()

## 2. Dónde está cada zona: posiciones 2D

Un transformer no sabe de geometría: para él, 240 tokens son una bolsa
desordenada. Por eso a cada token de imagen se le suma un vector de
**posición 2D** (senoidal, fijo), que codifica su fila y su columna.

La prueba de que codifica "cerca/lejos": la similitud del vector de posición
de una zona con el de todas las demás.

In [ ]:
with torch.no_grad():
    pos2d = model.encoder_cam_feat_pos_embed(features)[0]  # (192, 12, 20)
flat = pos2d.reshape(pos2d.shape[0], -1).T
flat = flat / flat.norm(dim=1, keepdim=True)
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, (r, c) in zip(axes, [(2, 3), (6, 10), (10, 17)]):
    similarity = (flat @ flat[r * cols + c]).reshape(rows, cols)
    ax.imshow(similarity, cmap="viridis"); ax.scatter([c], [r], c="r", marker="x")
    ax.set_title(f"similitud con la zona ({r}, {c})")
plt.tight_layout()

## 3. La lista completa: 242 tokens

| Token | De dónde sale | Posición |
|---|---|---|
| 1 de la `z` | 32 números × una matriz → 192 | aprendida |
| 1 del estado | 26 números × una matriz → 192 | aprendida |
| 240 de la imagen | la grilla de la ResNet → 192 | 2D senoidal fija |

(El paper: 1 + 1 + 1200 = 1202.)

In [ ]:
scene = act.scene_tokens(model, obs, latent)
print("tokens:", tuple(scene.tokens.shape), "→ (secuencia, lote, 192)")
print("antes de la imagen:", scene.n_prefix, "tokens · grilla:", scene.image_grid)

## 4. El encoder: 4 capas donde todos se miran

En cada capa:

1. **Atención**: cada token mira a todos los demás. Las posiciones se suman a
   las *queries* y *keys* (para decidir a quién mirar) pero **no** a los
   *values* (lo que se copia). Y se suman **en cada capa**, no sólo al
   principio.
2. **MLP** (192 → 768 → 192): cada token "procesa lo que juntó", por su cuenta.
3. Conexiones residuales y normalización después de cada parte (*post-norm*).

Mientras la atención mueve información **entre** tokens, la MLP transforma
**cada** token por separado. Es la división de tareas de cualquier
transformer.

¿A dónde mira el token del **estado**, capa por capa?

In [ ]:
with act.capture_attention(model) as weights:
    memory = act.encode_scene(model, scene)
print("memoria:", tuple(memory.shape))

fig, axes = plt.subplots(1, config.n_encoder_layers, figsize=(16, 2.8))
for layer, ax in enumerate(axes):
    row = weights[f"encoder.layers.{layer}.self_attn"][0, 1]  # fila del token del estado
    ax.imshow(image.permute(1, 2, 0))
    ax.imshow(row[scene.n_prefix:].reshape(rows, cols), cmap="inferno", alpha=0.6, extent=(0, 640, 360, 0))
    ax.set_title(f"capa {layer}: imagen {row[scene.n_prefix:].sum():.0%}"); ax.axis("off")
plt.suptitle("a dónde mira el token del estado", y=1.04); plt.tight_layout()

El estado lee casi todo de la imagen. A la salida del encoder, cada token ya
no es "sólo imagen" o "sólo estado": es una mezcla. Esa mezcla es la
**memoria** (242×192) que va a leer el decoder.

## 5. El decoder: 48 casilleros que leen la memoria

Un transformer da una salida por cada token que le entra. Necesitamos 48
acciones, así que se agregan **48 casilleros** (*queries*), uno por paso
futuro. Lo que realmente pasa en LeRobot:

- Los casilleros **arrancan en cero**. Lo único que los distingue es una
  **posición aprendida** (48×192) que se suma en **cada** atención: el
  casillero 0 aprendió a significar "ahora" y el 47 "dentro de 1,6 s".
- **Atención propia** entre los 48 casilleros.
- **Cross-attention** a la memoria:
  - *queries* = casilleros + su posición;
  - *keys* = memoria + las posiciones del encoder (para saber de qué zona viene
    cada token);
  - *values* = memoria, sin posiciones.
- MLP, normalización final y una capa lineal 192 → 26.

Nuestro decoder tiene **1 capa** (el bug del original, en L3.1).

### ¿Qué zona de la imagen lee cada paso futuro?

In [ ]:
with act.capture_attention(model) as weights:
    decoded, normalized = act.decode_chunk(model, memory, scene)
cross = weights["decoder.layers.0.multihead_attn"][0]  # (48, 242)
print(f"atención a la z {cross[:, 0].mean():.1%} · al estado {cross[:, 1].mean():.1%} · a la imagen {cross[:, 2:].sum(1).mean():.1%}")

steps = [0, 16, 32, 47]
fig, axes = plt.subplots(1, len(steps), figsize=(16, 2.8))
for ax, step in zip(axes, steps):
    ax.imshow(image.permute(1, 2, 0))
    ax.imshow(cross[step, scene.n_prefix:].reshape(rows, cols), cmap="inferno", alpha=0.6, extent=(0, 640, 360, 0))
    ax.set_title(f"paso {step} (t + {step / 30:.1f} s)"); ax.axis("off")
plt.tight_layout()

Dos cosas para mirar:

- **Casi no lee el token del estado** (0,1%). No es que ignore los brazos: el
  encoder ya metió el estado adentro de los tokens de imagen, así que leer
  "imagen" es leer también el estado.
- **Pasos distintos leen zonas distintas.** El presente y el futuro lejano no
  necesitan la misma información.

En este cuadro, los pasos lejanos leen sobre todo las manos y la prenda. El
paso 0, en cambio, mira mucho el borde de arriba, donde no hay nada de la
tarea. Una hipótesis (no verificada): la acción inmediata casi no necesita la
imagen, porque el estado actual ya está mezclado en todos los tokens, y la
atención se "estaciona" en zonas neutras. Probalo con otros cuadros.

(Ojo: son pesos de atención promediados entre las 4 cabezas y en una sola
capa. Dicen qué se lee, no "por qué".)

### Y entre los 48 casilleros

In [ ]:
self_attention = weights["decoder.layers.0.self_attn"][0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(self_attention, cmap="viridis"); axes[0].set_title("atención entre casilleros"); axes[0].set_xlabel("mira a"); axes[0].set_ylabel("casillero")
queries = model.decoder_pos_embed.weight.detach()
queries = queries / queries.norm(dim=1, keepdim=True)
axes[1].imshow(queries @ queries.T, cmap="RdBu", vmin=-1, vmax=1); axes[1].set_title("similitud entre las posiciones aprendidas")
plt.tight_layout()

- **La atención entre casilleros** es casi pareja (cerca de 1/48 cada uno):
  en nuestro modelo no aprendió una estructura marcada.
- **Las posiciones aprendidas** son casi independientes entre sí: fuera de la
  diagonal, la similitud ronda cero. La red no aprendió que "el paso 10 se
  parece al 11"; trata a cada paso como un casillero propio. Compará con las
  posiciones 2D fijas de la imagen (sección 2), que sí codifican cercanía.

## 6. La salida: 48 × 26

In [ ]:
actions = loaded.postprocessor(normalized)[0]
print("decoder:", tuple(decoded.shape), "→ cabeza lineal → normalizado", tuple(normalized.shape), "→ radianes", tuple(actions.shape))
with torch.no_grad():
    reference = loaded.postprocessor(policy.predict_action_chunk(obs))[0]
print("¿igual a predict_action_chunk?", torch.allclose(actions, reference, atol=1e-5))

names = plain.meta.features["action"]["names"][0]
start = int(plain.meta.episodes[int(frame["episode_index"])]["dataset_from_index"])
t = 3000 - start
truth = np.stack(plain.hf_dataset.with_format("numpy")[3000:3048]["action"])
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, joint in zip(axes, ["kRightShoulderPitch", "kRightElbow", "kRightHandIndex"]):
    j = names.index(joint)
    ax.plot(truth[:, j], color="0.6", lw=3, label="operador"); ax.plot(actions[:, j], label="ACT")
    ax.set_title(joint); ax.set_xlabel("paso del bloque")
axes[0].legend(); plt.tight_layout()

---

## Para profundizar: dónde están los parámetros

In [ ]:
parts = {"ResNet18": model.backbone, "encoder del VAE (sólo al entrenar)": model.vae_encoder,
         "encoder (4 capas)": model.encoder, "decoder (1 capa)": model.decoder}
counts = {name: sum(p.numel() for p in module.parameters()) for name, module in parts.items()}
counts["proyecciones, posiciones y cabeza"] = sum(p.numel() for p in model.parameters()) - sum(counts.values())
pd.Series(counts).apply(lambda n: f"{n / 1e6:.2f} M")

Más de dos tercios son la ResNet. El transformer que decide es chico: es la
versión liviana. Con los valores por defecto de LeRobot (tokens de 512, capa
interna de 3200) serían unos 52 M en total (L1).